# ViT APJN experiments notebook

This notebook follows the `vit-dyt` bootstrap flow from the reference Colab before running the APJN experiments.

In [ ]:
# Optional: install TeX/font packages for publication-style plots.
# Run this cell only if you want the LaTeX/NeurIPS-like font setup below.
!apt-get -qq update
!apt-get -qq install texlive-latex-extra texlive-fonts-recommended dvipng cm-super

In [ ]:
# Optional: configure NeurIPS-like LaTeX fonts.
# Run after the apt-get cell above.
# If you want the simpler Times-like fallback instead, use `configure_times_like_tex_fonts()`.
from vit_apjn_notebook_helpers import configure_neurips_like_tex_fonts, configure_times_like_tex_fonts

configure_times_like_tex_fonts()

In [ ]:
# Colab/local setup for vit-dyt.
from pathlib import Path
import subprocess
import sys

REPO_DIR = Path('vit-dyt')
INSTALL_TIMM = True
INSTALL_REQUIREMENTS = True
CLONE_IF_MISSING = True

if INSTALL_TIMM:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm'], check=True)

if not REPO_DIR.exists() and CLONE_IF_MISSING:
    subprocess.run(['git', 'clone', 'https://github.com/labofdoubt/vit-dyt', str(REPO_DIR)], check=True)

REQ = REPO_DIR / 'requirements.txt'
if INSTALL_REQUIREMENTS and REQ.exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REQ)], check=True)

if REPO_DIR.exists() and str(REPO_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(REPO_DIR.resolve()))

print('Repo directory:', REPO_DIR.resolve() if REPO_DIR.exists() else 'missing')

In [ ]:
# Common imports and helper bootstrap.
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

from vit_apjn_notebook_helpers import *

bootstrap_vit_dyt_repo(
    REPO_DIR,
    clone_if_missing=CLONE_IF_MISSING,
    install_requirements=INSTALL_REQUIREMENTS,
    install_timm=INSTALL_TIMM,
)
set_pub_style()
print('DEVICE =', DEVICE)

In [ ]:
# Global defaults and top-level tuning handles.
DEFAULT_DEPTH = 64
DEFAULT_ALPHAS = tuple(np.arange(0.1, 1.9 + 1e-9, 0.2).astype(float))
DEFAULT_Q0_GRID = tuple(np.linspace(0.0, 2.0, 41))
DEFAULT_P0_GRID = tuple(np.linspace(0.0, 2.0, 41))
DEFAULT_PRELN_SCALE_GRID = tuple(np.linspace(0.4, 8.0, 61))

MODEL_CFG = ModelConfig(
    model_name='vit_base_patch16_224',
    depth=DEFAULT_DEPTH,
    num_classes=100,
    img_size=224,
    replace_gelu_with_relu=True,
    inplace_relu=False,
    seed=0,
)
MEAN_FIELD_CFG = build_mean_field_cfg_for_vit_base(depth=MODEL_CFG.depth)
STYLE_CFG = FinalThreePanelStyleConfig(
    title_fs=18,
    label_fs=18,
    annotation_fs=18,
    colorbar_pad=0.04,
    )

# Direct plotting overrides for notebook figures.
PLOT_TICK_FS = 16
PLOT_LABEL_FS = 18
PLOT_TITLE_FS = 18
PLOT_ANNOTATION_FS = 18
PLOT_ALPHA_LEGEND_FS = 16

EQ_PANEL_WIDTH_RATIOS = (1.0, 1.5, 1.5)
EQ_PANEL_GAP_AB = 0.18
EQ_PANEL_GAP_BC = 0.18

print('MODEL_CFG =', cfg_to_dict(MODEL_CFG))

## 3. CIFAR repeated fits and random direct APJN points

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
PLOT_TICK_FS = 16
PLOT_LABEL_FS = 18
PLOT_TITLE_FS = 18
PLOT_ANNOTATION_FS = 18
PLOT_ALPHA_LEGEND_FS = 16

In [ ]:
MODEL_CFG = ModelConfig(
    model_name='vit_base_patch16_224',
    depth=64,
    num_classes=100,
    img_size=224,
    replace_gelu_with_relu=True,
    inplace_relu=False,
    seed=0,
)
MEAN_FIELD_CFG = build_mean_field_cfg_for_vit_base(depth=MODEL_CFG.depth)

In [ ]:

RESULTS_POSTFIX = ""
SAVE_RESULTS_ROOT = "/content/drive/MyDrive/ml_projects/mapes_variance"
SAVE_FIT_HIST_RESULTS = True
SAVE_RANDOM_DIRECT_RESULTS = True
REWRITE = False

In [ ]:
# Plot.
# Load/prepare once, then re-plot from `fit_scatter_plot_data`.
ATTN_MULT = 2.0
MLP_MULT = 1.0

# fit_hist_bundle = f"/content/drive/MyDrive/ml_projects/mapes_variance/fit_hist_depth64_stride8_determ_seed_124_mask_False_num_inits_8/results.pkl"
# attn and mlp mults
fit_hist_bundle = f"/content/drive/MyDrive/ml_projects/mapes_variance/fit_hist_depth64_stride8_determ_seed_124_mask_False_num_inits_8_attn_mult_{ATTN_MULT}_mlp_mult_{MLP_MULT}/results.pkl"

# random_inverse_bundle = "/content/drive/MyDrive/ml_projects/mapes_variance/random_direct_depth32_layers5_test/results.pkl"
# random_direct_bundle = "/content/drive/MyDrive/ml_projects/mapes_variance/random_direct_depth32_layers5_test/results.pkl"
# random_inverse_bundle = "/content/drive/MyDrive/ml_projects/mapes_variance/random_inverse_depth64_layers8/results.pkl"
# random_direct_bundle = "/content/drive/MyDrive/ml_projects/mapes_variance/random_direct_depth64_layers8_rescaled_preln_weight_2/results.pkl"
random_inverse_bundle = "/content/drive/MyDrive/ml_projects/mapes_variance/random_inverse_depth64_layers8_less_alphas/results.pkl"
random_direct_bundle = "/content/drive/MyDrive/ml_projects/mapes_variance/random_direct_depth64_layers8_less_alphas/results.pkl"

fit_scatter_plot_data = prepare_fit_and_scatter_plot_data(
    fit_hist_bundle,
    random_inverse_bundle,
    random_direct_bundle,
)

In [ ]:
# for i in range(150):
#   print(fit_scatter_plot_data['fit_bundle']['samples'][i]['direct_fit']['preln_scale_C'])

In [ ]:
DEFAULT_ALPHAS = tuple(np.arange(0.1, 1.9 + 1e-9, 0.2).astype(float))

DEFAULT_Q0_GRID = tuple(np.linspace(0.0, 2.0, 201))
# DEFAULT_P0_GRID = tuple(np.union1d(np.linspace(-0.2, 0, 61), np.linspace(0, 1.6, 101)))
DEFAULT_RHO0_GRID = tuple(np.linspace(-0.4, 1.0, 151))
# DEFAULT_P0_GRID = tuple(np.linspace(0.0, 0.0, 1))
DEFAULT_PRELN_SCALE_GRID = tuple(np.geomspace(0.1, 30, 400))

In [ ]:
MEAN_FIELD_CFG = build_mean_field_cfg_for_vit_base(
    depth=MODEL_CFG.depth,
    attn_mult=ATTN_MULT,
    mlp_mult=MLP_MULT,
    )

fit_scatter_plot_data_refit = refit_fit_scatter_plot_data(
    fit_scatter_plot_data,
    MEAN_FIELD_CFG,
    n_tokens=196,
    q0_values=DEFAULT_Q0_GRID,
    rho0_values=DEFAULT_RHO0_GRID,
    c_values=DEFAULT_PRELN_SCALE_GRID,
    fit_pre_ln=True,
    mask_all_p_values=False,
)

In [ ]:
print(fit_scatter_plot_data_refit['fit_bundle']['direct_mape'][27])

In [ ]:
mapes = fit_scatter_plot_data_refit['fit_bundle']['direct_mape']
print(sum(mapes)/len(mapes))

In [ ]:
vals = [fit_scatter_plot_data_refit['fit_bundle']['samples'][i]['direct_fit']['q0'] for i in range(50)]
print(vals[40])

In [ ]:
vals = [fit_scatter_plot_data_refit['fit_bundle']['samples'][i]['direct_fit']['p0'] for i in range(50)]
print(vals[40])

In [ ]:
fig_diag = plot_inverse_fit_sample_diagnostics(
    fit_scatter_plot_data,
    sample_index=27,
    style_cfg=STYLE_CFG,
    apjn_direction='direct'
)

In [ ]:
restored_stats = collect_restored_cifar_block0_dot_stats(
    MODEL_CFG,
    batch_seed=124,
    sample_index=5,
    block_indices=[0, 16, 32, 48, 60],
    use_derf=True,
    alpha = 1.0,
    std_threshold=0.0,
    max_epochs_to_search=20,
)

In [ ]:
a = plot_restored_cifar_block0_dot_histograms(restored_stats)

In [ ]:
FIT_PANEL_COL_GAP = 0.18
FIT_PANEL_ROW_GAP = 0.16
FIT_LOWER_TO_CBAR_GAP = 0.1
FIT_SCATTER_POINT_ALPHA = 0.7
FIT_PERCENTILE_LABEL_ALPHA = 0.7
FIT_PERCENTILE_LABEL_FS = 12
PANEL_D_REDUCED_GAP = 0.2

STYLE_CFG = FinalThreePanelStyleConfig(
    title_fs=20,
    label_fs=20,
    annotation_fs=20,
    colorbar_pad=0.0,
    )

fig_hist_scatter = plot_fit_and_scatter_figure(
    fit_scatter_plot_data_refit,
    STYLE_CFG,
    panel_col_gap=FIT_PANEL_COL_GAP,
    panel_row_gap=FIT_PANEL_ROW_GAP,
    lower_row_to_colorbar_gap=FIT_LOWER_TO_CBAR_GAP,
    tick_fs=PLOT_TICK_FS,
    label_fs=PLOT_LABEL_FS,
    alpha_legend_fs=PLOT_ALPHA_LEGEND_FS,
    title_fs=PLOT_TITLE_FS,
    percentile_annotation_fs=FIT_PERCENTILE_LABEL_FS,
    percentile_annotation_alpha=FIT_PERCENTILE_LABEL_ALPHA,
    scatter_point_alpha=FIT_SCATTER_POINT_ALPHA,
    legend_width_scale=0.72,
    legend_height_scale=0.4
)

In [ ]:
from vit_apjn_notebook_helpers import plot_scatter_only_figure

fig_scatter = plot_scatter_only_figure(
    fit_scatter_plot_data,
    STYLE_CFG,
    alphas=[0.3, 1.1, 1.9],
    title_fs=24,
    label_fs=22,
    tick_fs=20,
    alpha_legend_fs=20,
    colorbar_width_scale=0.4,
    colorbar_height_scale=0.45,
    colorbar_x_shift=0.02,
    colorbar_y_shift=0.05,
    preln_rect_width=0.24,
    preln_rect_height=0.9,
    preln_label_x=1.14,
    scatter_point_alpha=0.75,
)

**MAPE 4-panel plots**

In [ ]:
MULT_PAIRS = [(1.0, 1.0), (2.0, 1.0), (1.0, 2.0), (2.0, 2.0)]
PANEL_LAYOUT = {
    (1.0, 2.0): (0, 0),
    (2.0, 2.0): (0, 1),
    (1.0, 1.0): (1, 0),
    (2.0, 1.0): (1, 1),
}

def _panel_title(mult_pair):
    mlp_mult, attn_mult = mult_pair
    return rf"$(\sigma_{{21}}, \sigma_{{OV}})=({mlp_mult**2 * 0.6:.1f},\ {attn_mult**2 * 0.3:.1f})$"

def _fit_hist_bundle_path(mult_pair):
    mlp_mult, attn_mult = mult_pair
    if mult_pair == (1.0, 1.0):
        return "/content/drive/MyDrive/ml_projects/mapes_variance/fit_hist_depth64_stride8_determ_seed_124_mask_False_num_inits_8/results.pkl"
    return (
        "/content/drive/MyDrive/ml_projects/mapes_variance/"
        f"fit_hist_depth64_stride8_determ_seed_124_mask_False_num_inits_8_attn_mult_{attn_mult}_mlp_mult_{mlp_mult}/results.pkl"
    )

def _load_refit_plot_data(mult_pair):
    mlp_mult, attn_mult = mult_pair
    fit_scatter_plot_data = prepare_fit_and_scatter_plot_data(
        _fit_hist_bundle_path(mult_pair),
        random_inverse_bundle,
        random_direct_bundle,
    )
    mean_field_cfg = build_mean_field_cfg_for_vit_base(
        depth=MODEL_CFG.depth,
        attn_mult=attn_mult,
        mlp_mult=mlp_mult,
    )
    fit_scatter_plot_data_refit = refit_fit_scatter_plot_data(
        fit_scatter_plot_data,
        mean_field_cfg,
        n_tokens=196,
        q0_values=DEFAULT_Q0_GRID,
        rho0_values=DEFAULT_RHO0_GRID,
        c_values=DEFAULT_PRELN_SCALE_GRID,
        fit_pre_ln=True,
        mask_all_p_values=False,
    )
    print('Loaded and refitted')
    return fit_scatter_plot_data_refit

refit_plot_data_by_mult = {
    mult_pair: _load_refit_plot_data(mult_pair)
    for mult_pair in MULT_PAIRS
}

inverse_mape_pct_by_mult = {
    mult_pair: 100.0 * np.asarray(plot_data['fit_bundle']['inverse_mape'], dtype=float)
    for mult_pair, plot_data in refit_plot_data_by_mult.items()
}
direct_mape_pct_by_mult = {
    mult_pair: 100.0 * np.asarray(plot_data['fit_bundle']['direct_mape'], dtype=float)
    for mult_pair, plot_data in refit_plot_data_by_mult.items()
}

inverse_ylim_top = 1.05 * max(np.nanmax(vals) for vals in inverse_mape_pct_by_mult.values())
direct_ylim_top = 1.05 * max(np.nanmax(vals) for vals in direct_mape_pct_by_mult.values())

In [ ]:
MAPE_PANEL_TITLE_FS = 18
MAPE_LABEL_FS = 20
MAPE_TICK_FS = 20
MAPE_SECTION_TITLE_FS = 24
MAPE_SECTION_TITLE_Y = 1.05
MAPE_SUBPLOTS_TOP = 0.92

INVERSE_PANEL_COLORS = {
    (1.0, 1.0): "#4C6EF5",  # cobalt
    (2.0, 1.0): "#4C6EF5",  # cyan-teal
    (1.0, 2.0): "#4C6EF5",  # vivid indigo
    (2.0, 2.0): "#4C6EF5",  # electric blue
}
DIRECT_PANEL_COLORS = {
    (1.0, 1.0): "#2F9E44",  # emerald
    (2.0, 1.0): "#2F9E44",  # aqua green
    (1.0, 2.0): "#2F9E44",  # lime-olive
    (2.0, 2.0): "#2F9E44",  # deep mint
}

def _swarm_x_positions(n_points: int, center: float, width: float, rng: np.random.Generator):
    if n_points <= 0:
        return np.empty(0, dtype=float)
    if n_points == 1:
        return np.asarray([center], dtype=float)
    offsets = rng.uniform(-0.5 * width, 0.5 * width, size=int(n_points))
    order = np.argsort(offsets)
    return center + offsets[order]

def _plot_mape_panel(ax, mape_values, *, color, title):
    mape_pct = 100.0 * np.asarray(mape_values, dtype=float)
    x_vals = _swarm_x_positions(
        mape_pct.size,
        center=0.0,
        width=0.42,
        rng=np.random.default_rng(0),
    )

    ax.scatter(
        x_vals,
        mape_pct,
        color=color,
        s=34,
        alpha=0.82,
        edgecolors="white",
        linewidths=0.45,
        zorder=3,
    )

    for percentile, label in [(90, r"$90\%$ of data points")]:
        y_val = float(np.nanpercentile(mape_pct, percentile))
        ax.axhline(y_val, color=color, ls="--", lw=1.6, alpha=0.9, zorder=2)
        ax.text(
            0.98,
            y_val,
            label,
            ha="right",
            va="bottom",
            transform=ax.get_yaxis_transform(),
            fontsize=FIT_PERCENTILE_LABEL_FS,
            alpha=float(FIT_PERCENTILE_LABEL_ALPHA),
            color=color,
        )

    ax.set_title(title, fontsize=MAPE_PANEL_TITLE_FS)
    ax.set_xticks([])
    prettify_axes(ax)
    ax.tick_params(labelsize=MAPE_TICK_FS)
    return mape_pct


fig_mape_grid = plt.figure(figsize=(15, 6.6))
gs = fig_mape_grid.add_gridspec(
    2,
    5,
    width_ratios=[1.0, 1.0, 0.18, 1.0, 1.0],
    hspace=0.32,
    wspace=0.25,
)

inverse_axes = {
    (1.0, 2.0): fig_mape_grid.add_subplot(gs[0, 0]),
    (2.0, 2.0): fig_mape_grid.add_subplot(gs[0, 1]),
    (1.0, 1.0): fig_mape_grid.add_subplot(gs[1, 0]),
    (2.0, 1.0): fig_mape_grid.add_subplot(gs[1, 1]),
}
direct_axes = {
    (1.0, 2.0): fig_mape_grid.add_subplot(gs[0, 3]),
    (2.0, 2.0): fig_mape_grid.add_subplot(gs[0, 4]),
    (1.0, 1.0): fig_mape_grid.add_subplot(gs[1, 3]),
    (2.0, 1.0): fig_mape_grid.add_subplot(gs[1, 4]),
}

for mult_pair, ax in inverse_axes.items():
    _plot_mape_panel(
        ax,
        refit_plot_data_by_mult[mult_pair]['fit_bundle']['inverse_mape'],
        color=INVERSE_PANEL_COLORS[mult_pair],
        title=_panel_title(mult_pair),
    )
    ax.set_ylim(0.0, inverse_ylim_top)

for mult_pair, ax in direct_axes.items():
    _plot_mape_panel(
        ax,
        refit_plot_data_by_mult[mult_pair]['fit_bundle']['direct_mape'],
        color=DIRECT_PANEL_COLORS[mult_pair],
        title=_panel_title(mult_pair),
    )
    ax.set_ylim(0.0, direct_ylim_top)

for mult_pair, ax in inverse_axes.items():
    row_idx, col_idx = PANEL_LAYOUT[mult_pair]
    if col_idx == 0:
        ax.set_ylabel(r"MAPE, $\%$", fontsize=MAPE_LABEL_FS)
    if row_idx == 1:
        ax.set_xlabel("")

for mult_pair, ax in direct_axes.items():
    row_idx, col_idx = PANEL_LAYOUT[mult_pair]
    if col_idx == 0:
        ax.set_ylabel(r"MAPE, $\%$", fontsize=MAPE_LABEL_FS)
    if row_idx == 1:
        ax.set_xlabel("")

fig_mape_grid.text(
    0.3,
    MAPE_SECTION_TITLE_Y,
    r"(a) Per-sample $\mathcal{J}^{\, B, b}$-fit MAPE",
    ha="center",
    va="top",
    fontsize=MAPE_SECTION_TITLE_FS,
)
fig_mape_grid.text(
    0.73,
    MAPE_SECTION_TITLE_Y,
    r"(b) Per-sample $\mathcal{J}^{\, b, 0}$-fit MAPE",
    ha="center",
    va="top",
    fontsize=MAPE_SECTION_TITLE_FS,
)
fig_mape_grid.subplots_adjust(top=MAPE_SUBPLOTS_TOP)
fig_mape_grid.savefig('mapes_distribution.pdf', bbox_inches="tight", dpi=600)

**Block-0 q/p histograms**

In [ ]:
HIST_SAMPLE_INDICES = [0, 1, 4, 5]
HIST_BATCH_SEED = 124
HIST_BLOCK_INDEX = 0

block0_hist_stats_by_sample = {
    sample_index: collect_restored_cifar_block0_dot_stats(
        MODEL_CFG,
        batch_seed=HIST_BATCH_SEED,
        sample_index=sample_index,
        block_indices=[HIST_BLOCK_INDEX],
        std_threshold=0.0,
        max_epochs_to_search=20,
        print_preview=False,
    )
    for sample_index in HIST_SAMPLE_INDICES
}

print(f"Loaded block-{HIST_BLOCK_INDEX} q/p stats for samples {HIST_SAMPLE_INDICES} (batch_seed={HIST_BATCH_SEED}).")


In [ ]:
HIST_FIGSIZE = (11.5, 9.0)
HIST_PANEL_TITLE_FS = 18
HIST_LABEL_FS = 16
HIST_TICK_FS = 14
HIST_OUTER_WSPACE = 0.22
HIST_OUTER_HSPACE = 0.28
HIST_INNER_HSPACE = 0.18
HIST_BINS = 40
HIST_Q_COLOR = "#4c78a8"
HIST_P_COLOR = "#f58518"

hist_sample_grid = [[0, 1], [4, 5]]
fig_block0_hist = plt.figure(figsize=HIST_FIGSIZE)
outer = fig_block0_hist.add_gridspec(
    2,
    2,
    wspace=HIST_OUTER_WSPACE,
    hspace=HIST_OUTER_HSPACE,
)

for row, sample_row in enumerate(hist_sample_grid):
    for col, sample_index in enumerate(sample_row):
        inner = outer[row, col].subgridspec(2, 1, hspace=HIST_INNER_HSPACE)
        ax_q = fig_block0_hist.add_subplot(inner[0, 0])
        ax_p = fig_block0_hist.add_subplot(inner[1, 0])

        stats = block0_hist_stats_by_sample[sample_index]
        q_vals = np.asarray(stats["token_sqnorm_over_d"], dtype=float)
        p_vals = np.asarray(stats["pairwise_dot_over_d"], dtype=float)

        ax_q.hist(q_vals, bins=HIST_BINS, color=HIST_Q_COLOR, alpha=0.88)
        ax_p.hist(p_vals, bins=HIST_BINS, color=HIST_P_COLOR, alpha=0.88)

        ax_q.set_title(f"Sample {sample_index}", fontsize=HIST_PANEL_TITLE_FS)
        ax_q.set_xlabel(r"$q = x_i^T x_i / d$", fontsize=HIST_LABEL_FS)
        ax_p.set_xlabel(r"$p = x_i^T x_j / d$", fontsize=HIST_LABEL_FS)

        if col == 0:
            ax_q.set_ylabel("count", fontsize=HIST_LABEL_FS)
            ax_p.set_ylabel("count", fontsize=HIST_LABEL_FS)

        prettify_axes(ax_q)
        prettify_axes(ax_p)
        ax_q.tick_params(labelsize=HIST_TICK_FS)
        ax_p.tick_params(labelsize=HIST_TICK_FS)

fig_block0_hist
